# 6.3 指令微调 (Instruction Fine-Tuning / SFT)

指令微调使预训练模型学会遵循指令生成符合期望格式的输出，是从"语言模型"到"助手模型"的关键步骤。

本节涵盖：
- 数据构造
- 多轮对话微调
- 多任务微调
- 系统提示微调

## 1. 数据构造

**目的**：创建高质量的指令-回复对

**基本原理**：通过Self-Instruct、人工编写、开源数据集整合等方式构建多样化的指令微调数据，数据质量比数量更重要。

**数据构造方法**：
- **Self-Instruct**：用强模型生成指令数据，种子指令→生成新指令→生成回复
- **人工编写**：专业标注人员编写高质量指令-回复对
- **开源数据整合**：整合已有的开源指令数据集
- **数据过滤**：去除低质量、重复、有毒的数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)

vocab_size = 100
d_model = 64

class SimpleLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 2, d_model*4, batch_first=True), 1
        )
        self.head = nn.Linear(d_model, vocab_size)
    def forward(self, x, mask=None):
        h = self.embed(x)
        h = self.transformer(h, mask=mask)
        return self.head(h)

def create_instruction_data(n_samples=8, max_len=16):
    instructions = []
    responses = []
    for _ in range(n_samples):
        instr = torch.randint(1, 40, (max_len,))
        resp = torch.randint(40, 80, (max_len,))
        instructions.append(instr)
        responses.append(resp)
    return instructions, responses

instructions, responses = create_instruction_data()

print('=== Instruction Data Construction ===')
print(f'Samples: {len(instructions)}')
print(f'Instruction tokens range: 1-39 ("user" vocabulary)')
print(f'Response tokens range: 40-79 ("assistant" vocabulary)')
print(f'\nSample instruction: {instructions[0][:6].tolist()}...')
print(f'Sample response: {responses[0][:6].tolist()}...')

SEP_TOKEN = 90
EOS_TOKEN = 99
combined = []
for instr, resp in zip(instructions, responses):
    seq = torch.cat([instr, torch.tensor([SEP_TOKEN]), resp, torch.tensor([EOS_TOKEN])])
    combined.append(seq)

print(f'\nCombined sequence (instruction + SEP + response + EOS):')
print(f'  Length: {len(combined[0])}')
print(f'  SEP at position {len(instructions[0])}, EOS at end')
print(f'\nKey: Only compute loss on response tokens (after SEP).')
print(f'Instruction tokens provide context but are not trained on.')

## 2. 多轮对话微调

**目的**：使模型具备多轮对话能力

**基本原理**：使用多轮对话数据训练模型，在计算loss时通常只对助手回复部分计算损失，用户输入部分不参与loss计算。

**对话格式**：
```
[INST] 用户问题1 [/INST] 助手回答1 [INST] 用户问题2 [/INST] 助手回答2
```

**Loss掩码**：仅对[/INST]之后、下一个[INST]之前的token计算损失

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

INST_START = 91
INST_END = 92

def create_multi_turn_dialogue(n_turns=3, turn_len=4):
    tokens = []
    loss_mask = []
    for turn in range(n_turns):
        tokens.append(INST_START)
        loss_mask.append(0)
        user_tokens = torch.randint(1, 40, (turn_len,)).tolist()
        tokens.extend(user_tokens)
        loss_mask.extend([0] * turn_len)
        tokens.append(INST_END)
        loss_mask.append(0)
        assistant_tokens = torch.randint(40, 80, (turn_len,)).tolist()
        tokens.extend(assistant_tokens)
        loss_mask.extend([1] * turn_len)
    return torch.tensor(tokens), torch.tensor(loss_mask)

tokens, loss_mask = create_multi_turn_dialogue(n_turns=3, turn_len=4)

print('=== Multi-Turn Dialogue Fine-Tuning ===')
print(f'Token sequence: {tokens.tolist()}')
print(f'Loss mask:      {loss_mask.tolist()}')
print(f'\n1 = compute loss, 0 = skip loss')

n_loss_tokens = loss_mask.sum().item()
n_total = len(loss_mask)
print(f'\nLoss tokens: {n_loss_tokens}/{n_total} ({n_loss_tokens/n_total:.1%})')
print(f'Only assistant responses contribute to the training loss.')

print(f'\nKey: Loss masking ensures the model learns to generate')
print(f'assistant responses, not to imitate user questions.')

## 3. 多任务微调与系统提示微调

**多任务微调**：将不同任务（问答、摘要、翻译、代码等）的指令数据混合训练，使模型具备通用的指令遵循能力。

**系统提示微调**：在对话中加入系统提示（system prompt）进行训练，使模型能根据系统指令调整行为风格和输出格式。

**多任务混合策略**：
- 按比例混合不同任务的数据
- 温度采样：调整不同任务的采样概率
- 动态调整：根据训练进度调整任务比例

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

tasks = {
    'QA': {'weight': 0.3, 'n_samples': 3000},
    'Summarization': {'weight': 0.2, 'n_samples': 2000},
    'Translation': {'weight': 0.2, 'n_samples': 2000},
    'Coding': {'weight': 0.15, 'n_samples': 1500},
    'Math': {'weight': 0.15, 'n_samples': 1500},
}

print('=== Multi-Task Fine-Tuning ===')
print(f'Task mixing strategy:')
total_weight = sum(t['weight'] for t in tasks.values())
for task, info in tasks.items():
    print(f'  {task:15s}: weight={info["weight"]:.0%}, samples={info["n_samples"]:,}')

system_prompts = {
    'helpful': 'You are a helpful assistant.',
    'concise': 'You are a concise assistant. Keep answers brief.',
    'creative': 'You are a creative assistant. Think outside the box.',
}

print(f'\n=== System Prompt Fine-Tuning ===')
for style, prompt in system_prompts.items():
    print(f'  [{style}]: "{prompt}"')

print(f'\nKey: Multi-task mixing gives the model broad instruction-following ability.')
print(f'System prompts let users control the model behavior style at inference time.')